# 02 - Features and leakage

The feature table is the part of this repo most likely to be wrong in a way that flatters
you. This notebook is the audit: confirm the timing convention, confirm nothing sees the
future, then look at what the model actually uses.

### The timing convention

Each row is a **forecast origin** `(sku, origin_date)`.

* Dynamic features use data from `origin_date` or earlier. `units_lag_0` is the origin
  day's own units -- which you genuinely know when you sit down to forecast.
* `roll_mean_28` is the mean over the 28 days **ending on** the origin date, inclusive.
* Targets live in separate columns: `y_h1 ... y_h28` hold units on
  `origin_date + 1 ... + 28`, and `y_cum7` / `y_cum28` hold the cumulative demand over
  the next 7 / 28 days -- which is what a replenishment decision actually needs.

A horizon-`h` model therefore trains on `(features at t -> y_h)` and cannot see day
`t + h`.

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from demandml import config as C
from demandml.features import feature_columns, load_features

pl.Config(tbl_rows=25, tbl_cols=12)
sns.set_theme(style="whitegrid")

lf = load_features("daily")
names = lf.collect_schema().names()
feats = feature_columns(lf)
print(f"{len(names)} columns | {len(feats)} usable model features")

## What kinds of features are there?

In [ ]:
import re
groups = {
    "target lags":      [f for f in feats if re.match(r"units_lag_", f)],
    "target rolling":   [f for f in feats if re.match(r"units_(mean|std|median|max|zero_rate)_", f)],
    "same-weekday":     [f for f in feats if "dow_mean" in f or "same_dow" in f],
    "ewm / ratios":     [f for f in feats if "ewm" in f or "ratio" in f or f.endswith("_cv_28")],
    "intermittency":    [f for f in feats if f.startswith(("adi_", "cv2_", "nonzero_", "days_since"))],
    "traffic / ads":    [f for f in feats if f.startswith(("sessions_", "ad_", "impressions_", "clicks_"))],
    "price / promo":    [f for f in feats if f.startswith(("kf_", "unit_price_", "discount_"))],
    "rank":             [f for f in feats if "rank" in f],
    "calendar":         [f for f in feats if f.startswith(("fourier_", "days_to_")) or f in
                         ("dow","dom","doy","iso_week","month","quarter","year","t_index",
                          "is_weekend","is_holiday","retail_event","is_month_start","is_month_end")],
    "family / cross":   [f for f in feats if f.startswith(("fam_", "sku_"))],
}
for k, v in groups.items():
    print(f"{k:18s} {len(v):4d}   {', '.join(v[:4])}{' ...' if len(v) > 4 else ''}")
print(f"\nunaccounted: {sorted(set(feats) - {x for v in groups.values() for x in v})}")

## Audit 1 - verify the timing by hand

Recompute a few features from the raw series with numpy and assert they match. If you
edit `features.py`, rerun this.

In [ ]:
d = (lf.filter(pl.col(C.SKU) == pl.col(C.SKU).first()).collect().sort(C.DATE))
u = d.get_column(C.TARGET).to_numpy().astype(float)
sku = d.get_column(C.SKU)[0]
print(f"SKU {sku}, {len(d)} origins\n")

def check(label, got, want, skip=1):
    got, want = np.asarray(got, float)[skip:-30], np.asarray(want, float)[skip:-30]
    m = np.isfinite(got) & np.isfinite(want)
    print(f"{'PASS' if np.allclose(got[m], want[m], atol=1e-3) else 'FAIL'}  {label}")

check("units_lag_0 == units[t]",              d['units_lag_0'], u)
check("units_lag_1 == units[t-1]",            d['units_lag_1'][1:], u[:-1])
check("units_lag_7 == units[t-7]",            d['units_lag_7'][7:], u[:-7])
check("units_mean_7 == mean(units[t-6..t])",  d['units_mean_7'][6:], np.convolve(u, np.ones(7)/7, 'valid'))
check("y_h1 == units[t+1]",                   d['y_h1'][:-1], u[1:])
check("y_h7 == units[t+7]",                   d['y_h7'][:-7], u[7:])
check("y_cum7 == sum(units[t+1..t+7])",       d['y_cum7'][:-7], np.convolve(u[1:], np.ones(7), 'valid'))

## Audit 2 - the blunt leakage check

No legitimate feature should correlate ~1.0 with a future target. Anything above ~0.99
here is a bug, not a discovery.

In [ ]:
sub = lf.select(feats + ["y_h1"]).head(300_000).collect().drop_nulls("y_h1")
y = sub.get_column("y_h1").to_numpy().astype(float)
cors = []
for c in feats:
    s = sub.get_column(c)
    if not s.dtype.is_numeric():
        continue
    x = s.fill_null(0).to_numpy().astype(float)
    if np.std(x) > 1e-9:
        cors.append((abs(np.corrcoef(x, y)[0, 1]), c))
cors.sort(reverse=True)

print("top |corr| with y_h1:")
for r, c in cors[:12]:
    print(f"  {r:.3f}  {c}")
suspects = [c for r, c in cors if r > 0.99]
print(f"\nfeatures with |corr| > 0.99 (investigate these): {suspects or 'none'}")

## Audit 3 - null structure

Long lags are null early in each SKU's life. That is correct and LightGBM handles it
natively -- but if a feature is >90% null it is costing you more in noise than it pays
in signal.

In [ ]:
nulls = (lf.head(500_000)
           .select([pl.col(c).null_count().alias(c) for c in feats])
           .collect().transpose(include_header=True, column_names=["nulls"])
           .with_columns((100 * pl.col("nulls") / 500_000).round(1).alias("null_pct"))
           .sort("null_pct", descending=True))
display(nulls.head(20))
print("features >90% null:", nulls.filter(pl.col("null_pct") > 90).get_column("column").to_list())

## Feature importance

Fit one quick model and look at what it leans on. A raw same-day column near the top
would mean something escaped `feature_columns()`.

In [ ]:
from demandml.experiment import RunConfig, run

res = run(RunConfig(models=["lgbm_tweedie"], horizon=7, sku_limit=150,
                    splits=C.SplitSpec(n_folds=1, test_size=28)), save=False)
imp = pl.DataFrame(res.importances["lgbm_tweedie"])
display(imp.head(25))

fig, ax = plt.subplots(figsize=(8, 9))
top = imp.head(30)
sns.barplot(y=top.get_column("feature"), x=top.get_column("gain"), ax=ax)
ax.set(title="LightGBM gain, top 30", xlabel="gain")
plt.tight_layout()

## Where to go next on features

Things worth trying, roughly in order of expected payoff:

1. **Relative target.** Train on `units / units_mean_28` instead of raw units, then
   multiply the prediction back. A single model then does not have to learn one
   scale per SKU. Use `models.gbdt.ScaledTarget`.
2. **Price elasticity.** `kf_price_vs_91d` is in there; interactions between it and
   `is_deal` are not. Promo-response is usually the largest controllable driver.
3. **Cross-SKU cannibalisation.** When a SKU's sibling in `sku_family` goes on deal,
   this SKU's demand drops. `fam_*` features get part of the way; explicit
   "sibling on deal" flags would get further.
4. **Lead-time-aware targets.** If replenishment lead time is 45 days, `y_cum28` is the
   wrong question -- add a `y_cum45`.
5. **Drop features.** A few hundred features on intermittent data is a lot of noise.
   Try the top 60 by gain and see if it gets *better*.